# Task 3 — Silver Layer: Cleanse and Validate

Cleanses and validates all eight Bronze tables into Silver Delta tables with parallel quarantine tables (rejection_reason attached to every failing row), runs the Bronze–Silver–Quarantine reconciliation, and produces a data-quality report in the ops schema.

In [0]:
# -----------------------------------------------------------------------------
# Silver layer engine: config + generic validation framework
# -----------------------------------------------------------------------------
from functools import reduce
from pyspark.sql import functions as F, Window

try:
    catalog = dbutils.widgets.get("catalog_name")
except Exception:
    dbutils.widgets.text("catalog_name", "dev")
    catalog = dbutils.widgets.get("catalog_name")

BRONZE = f"{catalog}.bronze"
SILVER = f"{catalog}.silver"
OPS = f"{catalog}.ops"
DATE_MIN = F.lit("1994-01-01").cast("date")
DATE_MAX = F.lit("1998-12-31").cast("date")
META_COLS = ["source_file", "_ingested_at", "pipeline_run_id"]

dq_rows = []   # collected per rule: (table, code, description, failing_rows, action)


def date_in_window(*cols):
    """True when any of the given date columns is null or outside the operational window."""
    out = None
    for c in cols:
        bad = F.col(c).isNull() | (F.col(c) < DATE_MIN) | (F.col(c) > DATE_MAX)
        out = bad if out is None else (out | bad)
    return out


def fk_condition(col_name, ref_table, ref_col):
    """True when col_name is not null but has no match in the (already validated) Silver parent.
    Parent tables are small, so a collected distinct key list is cheap and avoids mutating df."""
    vals = [r[0] for r in spark.table(ref_table).select(ref_col).distinct().collect() if r[0] is not None]
    return F.col(col_name).isNotNull() & ~F.col(col_name).isin(vals) if vals else F.col(col_name).isNotNull()


def process_table(name, bronze_name, pk_cols, type_map, build_checks=None):
    """Cleanses one Bronze table into a typed Silver Delta table plus a quarantine table.

    pk_cols      : declared primary key columns
    type_map     : {business column: target type}; 'date' columns parsed as yyyy-M-d
    build_checks : optional fn(df) -> (df, [(code, cond, description)]) for FK/numeric/date rules
    Every Bronze row goes to exactly one destination, so bronze = silver + quarantine.
    """
    bdf = spark.table(f"{BRONZE}.{bronze_name}")
    typed = [F.try_to_date(F.col(c), "yyyy-M-d").alias(c) if t == "date" else F.col(c).cast(t).alias(c)
             for c, t in type_map.items()]
    df = bdf.select(*typed, *META_COLS)

    # -- 3.1 Primary key validation (deterministic keep-first ordering for duplicates) --
    w = Window.partitionBy(*pk_cols).orderBy(*[F.col(c) for c in type_map],
                                             F.col("_ingested_at"), F.col("source_file"))
    df = df.withColumn("_rn", F.row_number().over(w))
    null_pk_cond = reduce(lambda a, b: a | b, [F.col(c).isNull() for c in pk_cols])
    checks = [
        ("NULL_PK", null_pk_cond, f"PK ({', '.join(pk_cols)}) must be not null"),
        ("DUPLICATE_PK", (F.col("_rn") > 1) & ~null_pk_cond,
         f"PK ({', '.join(pk_cols)}) must be unique across every row"),
    ]

    # -- 3.2/3.3/3.4 table-specific checks (FK resolves against parent Silver tables) --
    if build_checks is not None:
        df, extra = build_checks(df)
        checks += extra

    bronze_count = bdf.count()
    for code, cond, desc in checks:
        failing = df.filter(cond).count()
        dq_rows.append((name, code, desc, failing, "QUARANTINE"))

    # -- 3.5 Quarantine split: ALL failing rule codes carried per row --
    reasons = F.concat_ws(",", *[F.when(cond, F.lit(rule_code)) for rule_code, cond, _ in checks])
    df = df.withColumn("_rejection_reason", reasons)
    out_cols = [F.col(c) for c in type_map] + [F.col(c) for c in META_COLS]
    silver = df.filter(F.col("_rejection_reason") == "").select(*out_cols)
    quarantine = (df.filter(F.col("_rejection_reason") != "")
                    .select(*out_cols, F.col("_rejection_reason").alias("rejection_reason")))

    silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.{name}")
    quarantine.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.{name}_quarantine")
    print(f"{name:15s} bronze={bronze_count:6d}  silver={silver.count():6d}  quarantine={quarantine.count():6d}")

In [0]:
# -----------------------------------------------------------------------------
# Reference tables (parent-before-child order): region -> nation -> airline, route
# -----------------------------------------------------------------------------

# --- region: 3.1 PK only (no dirty records expected; any violation is a critical fault) ---
process_table(
    "region", "region", ["r_regionkey"],
    {"r_regionkey": "int", "r_name": "string", "r_comment": "string"},
)

# --- nation: 3.1 PK not null + unique across all 25 rows ---
process_table(
    "nation", "nation", ["n_nationkey"],
    {"n_nationkey": "int", "n_name": "string", "n_regionkey": "int", "n_comment": "string"},
)

# --- airline: 3.1 PK + 3.4 FK al_nationkey -> silver.nation ---
def _airline_checks(df):
    return df, [
        ("INVALID_FK_NATION",
         fk_condition("al_nationkey", f"{SILVER}.nation", "n_nationkey"),
         "airline.al_nationkey must resolve to nation.n_nationkey"),
    ]

process_table(
    "airline", "airline", ["al_airlineid"],
    {"al_airlineid": "int", "al_name": "string", "al_address": "string",
     "al_nationkey": "int", "al_phone": "string", "al_rating": "decimal(10,2)",
     "al_comment": "string"},
    _airline_checks,
)

# --- route: 3.1 PK. The documented rt_originkey/rt_destinationkey FK checks are NOT
# applicable: the Bronze route table has no origin/destination nation key columns
# (origin/destination are embedded in ro_name, e.g. 'PEK-ADD'). Recorded in the DQ report. ---
process_table(
    "route", "route", ["ro_routeid"],
    {"ro_routeid": "int", "ro_name": "string", "ro_aircraft": "string",
     "ro_airline_brand": "string", "ro_routetype": "string", "ro_capacity": "int",
     "ro_cabinconfig": "string", "ro_baseprice": "decimal(10,2)", "ro_comment": "string"},
)
dq_rows.append(("route", "INVALID_FK_ORIGIN_NATION",
                "route.rt_originkey must resolve to nation.n_nationkey (not applicable: column absent in Bronze schema)",
                0, "NOT_APPLICABLE"))
dq_rows.append(("route", "INVALID_FK_DEST_NATION",
                "route.rt_destinationkey must resolve to nation.n_nationkey (not applicable: column absent in Bronze schema)",
                0, "NOT_APPLICABLE"))

In [0]:
# -----------------------------------------------------------------------------
# airline_route and passengers (parents already validated)
# -----------------------------------------------------------------------------

# --- airline_route: 3.1 composite PK + 3.2 numeric + 3.4 FKs ---
def _airline_route_checks(df):
    return df, [
        ("INVALID_FK_AIRLINE",
         fk_condition("ar_airlineid", f"{SILVER}.airline", "al_airlineid"),
         "airline_route.ar_airlineid must resolve to airline.al_airlineid"),
        ("INVALID_FK_ROUTE",
         fk_condition("ar_routeid", f"{SILVER}.route", "ro_routeid"),
         "airline_route.ar_routeid must resolve to route.ro_routeid"),
        ("INVALID_NUMERIC_OPERATINGCOST",
         F.col("ar_operatingcost").isNull() | (F.col("ar_operatingcost") <= 0),
         "airline_route.ar_operatingcost must be strictly greater than zero"),
    ]

process_table(
    "airline_route", "airline_route", ["ar_airlineid", "ar_routeid"],
    {"ar_airlineid": "int", "ar_routeid": "int", "ar_available_seats": "int",
     "ar_operatingcost": "decimal(10,2)", "ar_comment": "string"},
    _airline_route_checks,
)

# --- passengers: 3.1 PK + 3.4 FK pa_nationkey -> silver.nation ---
def _passenger_checks(df):
    return df, [
        ("INVALID_FK_NATION",
         fk_condition("pa_nationkey", f"{SILVER}.nation", "n_nationkey"),
         "passenger.pa_nationkey must resolve to nation.n_nationkey"),
    ]

process_table(
    "passengers", "passengers", ["pa_passengerid"],
    {"pa_passengerid": "int", "pa_name": "string", "pa_address": "string",
     "pa_nationkey": "int", "pa_phone": "string", "pa_loyaltypoints": "decimal(12,2)",
     "pa_travelpurpose": "string", "pa_comment": "string"},
    _passenger_checks,
)

In [0]:
# -----------------------------------------------------------------------------
# booking and flight_booking (the core booking fact chain, parents already validated)
# -----------------------------------------------------------------------------

# --- booking: 3.1 PK + 3.3 date window + 3.4 FK bk_passengerid -> silver.passengers ---
def _booking_checks(df):
    return df, [
        ("INVALID_FK_PASSENGER",
         fk_condition("bk_passengerid", f"{SILVER}.passengers", "pa_passengerid"),
         "booking.bk_passengerid must resolve to passenger.pa_passengerid"),
        ("INVALID_DATE_RANGE",
         date_in_window("bk_bookingdate", "bk_flightdate"),
         "booking dates (bk_bookingdate, bk_flightdate) must fall between 1994-01-01 and 1998-12-31"),
    ]

process_table(
    "booking", "booking", ["bk_bookingid"],
    {"bk_bookingid": "int", "bk_passengerid": "int", "bk_status": "string",
     "bk_totalprice": "decimal(12,2)", "bk_bookingdate": "date", "bk_flightdate": "date",
     "bk_bookingpriority": "string", "bk_agent": "string", "bk_comment": "string"},
    _booking_checks,
)

# --- flight_booking: 3.1 composite PK + 3.2 numerics + 3.3 window and cross-column
#     consistency + 3.4 FKs (booking, route, airline) ---
def _flight_booking_checks(df):
    # Cross-column check needs the parent booking date: bring it in as a helper column
    # (excluded from the Silver/Quarantine output schemas by process_table).
    parent = (spark.table(f"{SILVER}.booking")
              .select(F.col("bk_bookingid").alias("_pbid"),
                      F.col("bk_bookingdate").alias("_bk_bookingdate")))
    df = df.join(parent, F.col("fb_bookingid") == F.col("_pbid"), "left").drop("_pbid")

    dep_before_book = (F.col("fb_departuredate").isNotNull() & F.col("_bk_bookingdate").isNotNull()
                       & (F.col("fb_departuredate") < F.col("_bk_bookingdate")))
    arr_before_dep = (F.col("fb_arrivaldate").isNotNull() & F.col("fb_departuredate").isNotNull()
                      & (F.col("fb_arrivaldate") < F.col("fb_departuredate")))

    return df, [
        ("INVALID_FK_BOOKING",
         fk_condition("fb_bookingid", f"{SILVER}.booking", "bk_bookingid"),
         "flight_booking.fb_bookingid must resolve to booking.bk_bookingid"),
        ("INVALID_FK_ROUTE",
         fk_condition("fb_routeid", f"{SILVER}.route", "ro_routeid"),
         "flight_booking.fb_routeid must resolve to route.ro_routeid"),
        ("INVALID_FK_AIRLINE",
         fk_condition("fb_airlineid", f"{SILVER}.airline", "al_airlineid"),
         "flight_booking.fb_airlineid must resolve to airline.al_airlineid"),
        ("INVALID_NUMERIC_TICKETPRICE",
         F.col("fb_ticketprice").isNull() | (F.col("fb_ticketprice") <= 0),
         "flight_booking.fb_ticketprice must be strictly greater than zero"),
        ("INVALID_NUMERIC_DISCOUNT",
         F.col("fb_discount").isNull() | (F.col("fb_discount") < 0) | (F.col("fb_discount") > 1),
         "flight_booking.fb_discount must lie in the closed interval [0, 1]"),
        ("INVALID_NUMERIC_SEATS",
         F.col("fb_seats").isNull() | (F.col("fb_seats") < 1),
         "flight_booking.fb_seats must be a positive integer (1 or more)"),
        ("INVALID_DATE_RANGE",
         date_in_window("fb_departuredate", "fb_commitdate", "fb_arrivaldate"),
         "flight_booking dates (departure, commit, arrival) must fall between 1994-01-01 and 1998-12-31"),
        ("INVALID_DATE_DEPARTURE_BEFORE_BOOKING", dep_before_book,
         "flight_booking.fb_departuredate must be on or after the parent bk_bookingdate"),
        ("INVALID_DATE_ARRIVAL_BEFORE_DEPARTURE", arr_before_dep,
         "flight_booking.fb_arrivaldate must be on or after fb_departuredate"),
    ]

process_table(
    "flight_booking", "flight_booking", ["fb_bookingid", "fb_linenumber"],
    {"fb_bookingid": "int", "fb_routeid": "int", "fb_airlineid": "int", "fb_linenumber": "int",
     "fb_seats": "int", "fb_ticketprice": "decimal(10,2)", "fb_discount": "decimal(5,4)",
     "fb_tax": "decimal(5,4)", "fb_refundflag": "string", "fb_segmentstatus": "string",
     "fb_departuredate": "date", "fb_commitdate": "date", "fb_arrivaldate": "date",
     "fb_cabinclass": "string", "fb_comment": "string"},
    _flight_booking_checks,
)

In [0]:
# -----------------------------------------------------------------------------
# 3.5 Bronze-Silver-Quarantine reconciliation (re-reads the stored tables)
# -----------------------------------------------------------------------------
from pyspark.sql import types as T

table_names = ["region", "nation", "airline", "route", "airline_route",
               "passengers", "booking", "flight_booking"]

recon_rows = []
for name in table_names:
    bronze_n = spark.table(f"{BRONZE}.{name}").count()
    silver_n = spark.table(f"{SILVER}.{name}").count()
    quar_n = spark.table(f"{SILVER}.{name}_quarantine").count()
    balanced = bronze_n == silver_n + quar_n
    recon_rows.append((name, bronze_n, silver_n, quar_n, balanced))

recon_schema = T.StructType([
    T.StructField("table", T.StringType()),
    T.StructField("bronze_rows", T.LongType()),
    T.StructField("silver_rows", T.LongType()),
    T.StructField("quarantine_rows", T.LongType()),
    T.StructField("balanced", T.BooleanType()),
])
recon_df = spark.createDataFrame(recon_rows, recon_schema)
display(recon_df)

unbalanced = [r["table"] for r in recon_rows if not r[4]]
if unbalanced:
    raise Exception(f"PIPELINE DEFECT: Bronze = Silver + Quarantine reconciliation FAILED for: {unbalanced}")
print("Reconciliation PASSED: Bronze = Silver + Quarantine for all 8 tables.")

In [0]:
# -----------------------------------------------------------------------------
# 3.5 Data-quality report: one row per check per table (zero-fail rules included)
# -----------------------------------------------------------------------------
dq_schema = T.StructType([
    T.StructField("table_name", T.StringType()),
    T.StructField("rule_code", T.StringType()),
    T.StructField("check_description", T.StringType()),
    T.StructField("failing_rows", T.LongType()),
    T.StructField("action", T.StringType()),
])
dq_df = spark.createDataFrame(dq_rows, dq_schema)

dq_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OPS}.dq_report")
print(f"Data-quality report written to {OPS}.dq_report ({dq_df.count()} checks recorded).")
display(dq_df.orderBy("table_name", "rule_code"))